In [12]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

In [31]:
train=pd.read_csv('/Users/taha/Downloads/playground-series-s4e6/train.csv')
test=pd.read_csv('/Users/taha/Downloads/playground-series-s4e6/test.csv')

In [3]:
pip install catboost lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.8/28.8 MB 1.0 MB/s  0:00:28 eta 0:00:01 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 2.1 MB/s  0:00:00m 1.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [catboost]━━━━━━━━━ 2/3 [catboost]
Note: you may need to restart the kernel to use updated packages.


In [21]:
X = train.drop(columns=["id", "Target"])
y = train["Target"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [22]:
print("Proportions y_train :\n", y_train.value_counts(normalize=True))
print("10 premiers indices X_train :", X_train.index[:10])

Proportions y_train :
 Target
Graduate    0.474156
Dropout     0.330594
Enrolled    0.195249
Name: proportion, dtype: float64
10 premiers indices X_train : Index([75923, 50969, 57555, 62378, 41157, 19814, 47138, 62292, 59664, 75773], dtype='int64')


In [23]:
X.nunique()

Marital status                                       6
Application mode                                    22
Application order                                    8
Course                                              19
Daytime/evening attendance                           2
Previous qualification                              21
Previous qualification (grade)                     110
Nacionality                                         18
Mother's qualification                              35
Father's qualification                              39
Mother's occupation                                 40
Father's occupation                                 56
Admission grade                                    668
Displaced                                            2
Educational special needs                            2
Debtor                                               2
Tuition fees up to date                              2
Gender                                               2
Scholarshi

In [24]:
cat_cols = [
    'Marital status', 'Application mode', 'Course', 'Previous qualification', 
    'Nacionality', "Mother's qualification", "Father's qualification", 
    "Mother's occupation", "Father's occupation", 'Displaced', 
    'Educational special needs', 'Debtor', 'Tuition fees up to date', 
    'Gender', 'Scholarship holder', 'International'
]

In [25]:
for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')

In [26]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)
# Ordre alphabétique : Dropout=0, Enrolled=1, Graduate=2
print(dict(zip(le.classes_, le.transform(le.classes_)))) 

{'Dropout': np.int64(0), 'Enrolled': np.int64(1), 'Graduate': np.int64(2)}


In [29]:
from catboost import CatBoostClassifier

# 1. Instanciation du modèle
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    loss_function='MultiClass',
    random_seed=42,
    verbose=0,
    cat_features=cat_cols # Si vous avez défini vos cat_cols comme vu à l'étape précédente
)

# 2. Vous pouvez ensuite lancer votre validation croisée avec ce modèle

In [30]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accuracies = []

# Boucle sur les 5 plis
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    # Séparation des données pour ce pli
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Initialisation et entraînement du modèle
    # (Exemple : model.fit(X_fold_train, y_fold_train))
    model.fit(X_fold_train, y_fold_train)
    
    # Prédiction et évaluation
    preds = model.predict(X_fold_val)
    acc = accuracy_score(y_fold_val, preds)
    accuracies.append(acc)
    
    print(f"Pli {fold + 1} - Accuracy : {acc:.4f}")

print(f"\nAccuracy moyenne globale : {np.mean(accuracies):.4f} (+/- {np.std(accuracies):.4f})")

Pli 1 - Accuracy : 0.8267
Pli 2 - Accuracy : 0.8257
Pli 3 - Accuracy : 0.8326
Pli 4 - Accuracy : 0.8321
Pli 5 - Accuracy : 0.8270

Accuracy moyenne globale : 0.8288 (+/- 0.0029)


In [32]:
from sklearn.metrics import classification_report, confusion_matrix

# Entraînement sur tout X_train
model.fit(X_train, y_train)

# Prédiction sur X_val
val_preds = model.predict(X_val)

# Rapport de classification détaillé
print("Rapport de classification sur le jeu de validation :")
print(classification_report(y_val, val_preds))

Rapport de classification sur le jeu de validation :
              precision    recall  f1-score   support

     Dropout       0.90      0.82      0.86      5059
    Enrolled       0.65      0.60      0.63      2988
    Graduate       0.85      0.93      0.89      7257

    accuracy                           0.83     15304
   macro avg       0.80      0.79      0.79     15304
weighted avg       0.83      0.83      0.83     15304



In [33]:
# S'assurer que les colonnes catégorielles sont bien au format 'category' pour le test aussi
for col in cat_cols:
    test[col] = test[col].astype('category')

# Prédictions sur le test (en excluant la colonne 'id')
X_test = test.drop(columns=["id"], errors="ignore")
test_preds = model.predict(X_test)

# Si les prédictions sont sous forme de tableau 2D ou d'objets, on s'assure d'avoir un vecteur simple
if test_preds.ndim > 1:
    test_preds = test_preds.flatten()

# Création du DataFrame de soumission
submission = pd.DataFrame({
    'id': test['id'],
    'Target': test_preds
})

# Sauvegarde au format CSV
submission.to_csv('submission_catboost.csv', index=False)
print("Fichier de soumission généré avec succès !")
submission.head()

Fichier de soumission généré avec succès !


,id,Target
0,76518,Dropout
1,76519,Graduate
2,76520,Graduate
3,76521,Enrolled
4,76522,Enrolled
